# Housing Market Module Test

## Testing Basic Housing Supply, Demand, and Pricing Dynamics

This notebook validates the housing market module with:
- 🏠 **Housing Supply Tracking**: Inventory and new construction flows
- 💰 **Price Dynamics**: Market price adjustments based on supply-demand balance
- 🧮 **Calculator Components**: Demand calculations and affordability indices
- 🔗 **Multi-Module Dependencies**: Integration with population and economic modules
- 📊 **Market Visualizations**: Housing trends and affordability analysis

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✅ Libraries imported successfully!")

In [ ]:
# Add the sd_toolkit to the path
import sys
import os
sys.path.insert(0, os.path.join('..', '..', '..', 'sd_toolkit'))

# Import System Dynamics Toolkit components
from sd_toolkit.config import YAMLSystemBuilder, TemplateManager, rate_registry
from sd_toolkit.engine.system import SystemModel
from sd_toolkit.engine.elements import Stock, Flow, Auxiliary, Calculator
from sd_toolkit.data.loader import SpatioTemporalData
from sd_toolkit.analysis.plotting import SystemPlotter
from sd_toolkit.core.units import Q_, unit_registry

print("✅ System Dynamics Toolkit loaded successfully!")
print("🏠 Ready for housing market module testing")
print(f"📊 Available rate functions: {len(rate_registry.list_functions())}")

In [ ]:
# Load model structure and scenario parameters
print("📂 Loading housing market configuration...")

# Load model structure
model_structure_path = Path('model_structure.yaml')
with open(model_structure_path, 'r') as file:
    model_structure = yaml.safe_load(file)

print(f"✅ Model structure loaded: {model_structure['model']['name']}")

# Load scenario parameters
scenario_path = Path('scenario_parameters.yaml')
with open(scenario_path, 'r') as file:
    scenario_params = yaml.safe_load(file)

print(f"✅ Scenario parameters loaded: {scenario_params['scenario']['name']}")

# Display model dimensions
print("\n🏗️ Model Dimensions:")
for dim_name, dim_info in model_structure['dimensions'].items():
    print(f"  • {dim_name}: {dim_info['labels']} (size: {dim_info['size']})")

# Display external dependencies
print("\n🔗 External Dependencies:")
if 'external_dependencies' in model_structure:
    for dep in model_structure['external_dependencies']:
        print(f"  • {dep['name']}: from {dep['source_module']}.{dep['source_element']}")
else:
    print("  • No external dependencies defined")

In [ ]:
# Analyze initial housing market conditions
print("📊 Analyzing initial housing market conditions...")

# Extract initial housing data
housing_inventory = np.array(scenario_params['constants']['housing_inventory_distribution'])
housing_supply = np.array(scenario_params['constants']['housing_supply_distribution'])
housing_prices = np.array(scenario_params['constants']['average_housing_price_distribution'])
income_multipliers = np.array(scenario_params['constants']['income_demand_multiplier'])

housing_types = model_structure['dimensions']['housing_type']['labels']
price_brackets = model_structure['dimensions']['price_bracket']['labels']

print(f"\n🏠 Initial Housing Inventory:")
inventory_df = pd.DataFrame(housing_inventory, index=housing_types, columns=price_brackets)
print(inventory_df)

print(f"\n💰 Initial Housing Prices:")
prices_df = pd.DataFrame(housing_prices, index=housing_types, columns=price_brackets)
print(prices_df)

print(f"\n📈 Income Demand Multipliers:")
multipliers_df = pd.DataFrame(income_multipliers, index=housing_types, columns=price_brackets)
print(multipliers_df)

# Calculate summary statistics
total_inventory = housing_inventory.sum()
avg_price_affordable = housing_prices[:, 0].mean()
avg_price_market = housing_prices[:, 1].mean()
price_ratio = avg_price_market / avg_price_affordable

print(f"\n📋 Summary Statistics:")
print(f"  • Total Housing Inventory: {total_inventory:,} units")
print(f"  • Average Affordable Price: ${avg_price_affordable:,.0f}")
print(f"  • Average Market Rate Price: ${avg_price_market:,.0f}")
print(f"  • Market/Affordable Price Ratio: {price_ratio:.1f}x")

In [ ]:
# Create housing market visualizations
print("📊 Creating housing market visualizations...")

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Housing Market Module - Initial Conditions Analysis', fontsize=16, fontweight='bold')

# Plot 1: Housing inventory by type and price bracket
ax1 = axes[0, 0]
inventory_df.plot(kind='bar', ax=ax1, color=['lightblue', 'darkblue'])
ax1.set_title('Housing Inventory by Type and Price Bracket')
ax1.set_xlabel('Housing Type')
ax1.set_ylabel('Units')
ax1.legend(title='Price Bracket')
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Housing prices by type and price bracket
ax2 = axes[0, 1]
prices_df.plot(kind='bar', ax=ax2, color=['lightgreen', 'darkgreen'])
ax2.set_title('Average Housing Prices')
ax2.set_xlabel('Housing Type')
ax2.set_ylabel('Price (USD)')
ax2.legend(title='Price Bracket')
ax2.tick_params(axis='x', rotation=45)

# Plot 3: Income demand multipliers
ax3 = axes[1, 0]
multipliers_df.plot(kind='bar', ax=ax3, color=['orange', 'red'])
ax3.set_title('Income Demand Multipliers')
ax3.set_xlabel('Housing Type')
ax3.set_ylabel('Multiplier')
ax3.legend(title='Price Bracket')
ax3.tick_params(axis='x', rotation=45)

# Plot 4: Housing market value distribution
ax4 = axes[1, 1]
market_values = housing_inventory * housing_prices / 1e9  # Convert to billions
market_values_df = pd.DataFrame(market_values, index=housing_types, columns=price_brackets)
market_values_df.plot(kind='bar', ax=ax4, color=['purple', 'darkviolet'])
ax4.set_title('Housing Market Value Distribution')
ax4.set_xlabel('Housing Type')
ax4.set_ylabel('Market Value (Billion USD)')
ax4.legend(title='Price Bracket')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Housing market visualizations created")

In [ ]:
# Test model building (without external dependencies for now)
print("🔧 Testing housing market model building...")

# Create a simplified version for testing (without external dependencies)
test_config = model_structure.copy()
test_config['constants'] = scenario_params['constants']

# Remove external dependencies for standalone testing
if 'external_dependencies' in test_config:
    print("⚠️ Removing external dependencies for standalone testing")
    del test_config['external_dependencies']
    
    # Remove connections that depend on external elements
    external_deps = ['total_population', 'employed_population', 'average_income']
    test_config['connections'] = [
        conn for conn in test_config['connections'] 
        if conn['from'] not in external_deps
    ]

# Create YAML system builder
builder = YAMLSystemBuilder()

# Build the model
try:
    housing_model = builder.build_from_dict(test_config)
    print(f"✅ Model built successfully: {housing_model.name}")
    print(f"📊 Model elements: {len(housing_model.elements)}")
    print(f"📦 Stocks: {len(housing_model.stocks)}")
    print(f"🌊 Flows: {len(housing_model.flows)}")
    print(f"🧮 Calculators: {len(housing_model.calculators)}")
    print(f"🔧 Auxiliaries: {len(housing_model.auxiliaries)}")
    
except Exception as e:
    print(f"❌ Error building model: {e}")
    housing_model = None

# Model validation summary
print("\n📋 Housing Market Module Validation Summary")
print("=" * 55)

validation_results = {
    "Model Structure Loading": "✅ Success" if model_structure else "❌ Failed",
    "Scenario Parameters Loading": "✅ Success" if scenario_params else "❌ Failed",
    "Initial Data Analysis": "✅ Success",
    "Visualizations": "✅ Success",
    "Model Building (Standalone)": "✅ Success" if housing_model else "❌ Failed"
}

for test, status in validation_results.items():
    print(f"  {test}: {status}")

print("\n🎯 Next Steps:")
if housing_model:
    print("  • ✅ Housing market module structure is valid")
    print("  • 🔗 Ready for integration with population and economic modules")
    print("  • 🏗️ Can proceed with integrated model development")
else:
    print("  • ❌ Fix model building issues before proceeding")
    print("  • 🔍 Check parameter consistency and function definitions")
    print("  • 🛠️ Validate YAML syntax and dependency resolution")

print("\n🏁 Housing Market Module Test Complete!")